# Task axis vs. communication subspace: all ordered area pairs, one session

Is the alignment between an area's **task (context) axis** and its **communication
subspace** with another area a frontal-cortex speciality, or is it universal?

This notebook is a wrapper around the modules and scripts in `code/`:

- `task_axis_comm_subspace.py` -- the analysis (`analyse_session`). Full description in
  its module docstring. In short: quiescent-window firing rates, z-scored, previous-trial stimulus / response /
  reward, running speed and pupil area regressed out (`NUISANCE_REGRESSORS`, `none` to skip) and **no condition-mean
  subtraction**; QC-pass single units; areas with at
  least `MIN_UNITS` (30) units, every area subsampled to exactly 30 units,
  `N_SUBSAMPLES` (10) times; LDA task axis per area with balanced block-wise decoding accuracy
  (one block of each context held out, equal priors); reduced-rank-regression communication subspace per ordered pair with 10-fold
  CV R² vs. rank and the 1-SEM dimensionality rule; alignment = norm of the unit task
  axis projected onto the subspace, tested against a shuffled-label axis null (an LDA axis fit on
  shuffled labels, projected onto the same subspace), FDR-corrected over pairs; the random-axis chance
  is a reference. Decoding is judged against a block-permutation null. Every null draw is generated once
  per session and shared by all unit subsamples (2026-10-06).
- `make_alignment_figures.py` -- Figure 1 (task axis and communication subspace are
  meaningful), Figure 2 (alignment across all pairs vs. chance) per session, and pooled
  Figures 1 and 2 over every session that has results.
- `patch_cv_quality.py`, `patch_lobo_r2.py`, `alignment_controls.py`, `r2_dim_null.py` --
  cross-validated fit quality, the cross-block control and the rank-d R² null (see below).
- `make_controls_figure.py`, `make_report_figures.py` -- pooled control figure and the report
  figures (Figure 1, Figure 2, Supplementary Figure 1).

Reads `DATACUBE_ROOT`, `SESSION_ID` and `RESULTS_DIR` from the environment like the
other notebooks, so `run_capsule.py` runs it once per session; the pooled figure is
rebuilt at the end of every session run from all results present so far.

In [ ]:
import os, pickle, sys
from pathlib import Path

CODE_DIR = Path.cwd().parent
sys.path.insert(0, str(CODE_DIR))
import task_axis_comm_subspace as tac
import make_alignment_figures as maf

capsule_root = Path('/root/capsule/data/dynamicrouting_datacube')
root = Path(os.environ.get('DATACUBE_ROOT', capsule_root if capsule_root.exists() else CODE_DIR.parent / 'data' / 'dynamicrouting_datacube'))
session_id = os.environ.get('SESSION_ID', '743199_2024-12-05')
results_dir = Path(os.environ.get('RESULTS_DIR', '/root/capsule/results' if capsule_root.exists() else CODE_DIR.parent / 'results'))
out_dir = results_dir / session_id
out_dir.mkdir(parents=True, exist_ok=True)
print(f'DATACUBE_ROOT = {root}\nSESSION_ID = {session_id}\nresults -> {out_dir}')

# Analysis (about 6 minutes per session on a laptop)

In [ ]:
import pynwb
session = pynwb.read_nwb(tac.find_nwb(root, session_id))
try:
    res = tac.analyse_session(session.trials[:], session.units[:], session_id, session=session)
except RuntimeError as e:       # fewer than two areas with MIN_UNITS QC-pass units: nothing to analyse
    res = None
    print(f'SKIPPED: {e}')
if res is not None:
    with open(out_dir / 'alignment_results.pkl', 'wb') as f:
        pickle.dump(res, f)
    res['area_table'].to_csv(out_dir / 'task_axis_by_area.csv', index=False)
    res['pair_table'].to_csv(out_dir / 'alignment_by_pair.csv', index=False)


# Task axis per area

In [ ]:
res['area_table'].set_index('area').round(3) if res is not None else None


# Communication subspace and alignment per ordered pair

In [ ]:
cols = ['pair_type', 'r2', 'r2_q', 'dim', 'cos', 'cos_shuf_mean', 'cos_shuf_z', 'cos_shuf_q', 'cos_chance',
        'cos_block_hi', 'cos_tgt', 'cos_tgt_shuf_q', 'angle_deg', 'angle_chance_deg']
(res['pair_table'].set_index(['source', 'target'])[cols].round(3) if res is not None else None


In [ ]:
(res['pair_table'].groupby('pair_type')[['r2', 'dim', 'cos', 'cos_shuf_mean', 'cos_shuf_z', 'cos_chance', 'cos_tgt']].mean().round(3)) if res is not None else None


# Figures

Per-session Figures 1, 2 and Supplementary Figure 1 go to `<results>/<session>/figures/`. Then, from
every session that has results so far: pooled Figures 1-2 and the pooled cross-block control under
`<results>/pooled/`, and the report figures (Figure 1, Figure 2, Supplementary Figure 1) under
`<results>/report/figures/`. Because the capsule runs this notebook once per session, the pooled and
report figures written by the last session cover all sessions.


# Post-processing for the report figures (same session)

Three scripts in `code/` add to this session's results what the report figures need; each
rebuilds the same preprocessing, unit subsamples and folds as the run above:

- `patch_cv_quality.py` -- cross-validated (held-out blocks) predictive R² of the task axis per area and
  context decoding from the rank-d communication subspace per pair (with its block-permutation and trial-shuffle nulls).
- `patch_lobo_r2.py` -- leave-one-block-out R² of the rank-d communication subspace per pair.
- `alignment_controls.py` -- cross-block control: task axis fit on three blocks, communication
  subspace on the other three (`controls_results.pkl`).
- `r2_dim_null.py` -- trial-shuffle significance of the R² at the chosen dimensionality
  (`r2_dim_significance.csv`), used for the significance marks in Figure 1.

They run as subprocesses with this notebook's `DATACUBE_ROOT` / `RESULTS_DIR` / `SESSION_ID`.
Set `POST_STEPS` (comma-separated script names, or `none`) to change the list.


In [ ]:
import subprocess, time

if res is None:
    print('no analysis for this session; post-processing skipped')
else:
    default_steps = 'patch_cv_quality.py,patch_lobo_r2.py,alignment_controls.py,r2_dim_null.py'
    steps = [s.strip() for s in os.environ.get('POST_STEPS', default_steps).split(',') if s.strip() and s.strip() != 'none']
    env = dict(os.environ, DATACUBE_ROOT=str(root), RESULTS_DIR=str(results_dir), SESSION_ID=session_id,
               NUISANCE_REGRESSORS=','.join(res['nuisance_regressors']) or 'none')
    for script in steps:
        args = [] if script == 'alignment_controls.py' else [session_id]
        t0 = time.time()
        print(f'=== {script} ===', flush=True)
        subprocess.run([sys.executable, str(CODE_DIR / script), *args], check=True, env=env)
        print(f'=== {script} done in {(time.time() - t0) / 60:.1f} min ===\n', flush=True)

    with open(out_dir / 'alignment_results.pkl', 'rb') as f:   # reload: the patch scripts add fields in place
        res = pickle.load(f)


In [ ]:
from IPython.display import SVG, display
import make_controls_figure as mcf
import make_report_figures as mrf

if res is not None:
    fig_dir = out_dir / 'figures'
    maf.figure1(res, fig_dir)
    maf.supp_figure1(res, fig_dir)
    maf.figure2(res, fig_dir)
    display(SVG(str(fig_dir / f'figure1_{session_id}.svg')))
    display(SVG(str(fig_dir / f'figure2_{session_id}.svg')))

# pooled, controls and report figures over every session that has results so far
all_results = []
for p in sorted(results_dir.glob('*/alignment_results.pkl')):
    with open(p, 'rb') as f:
        all_results.append(pickle.load(f))
pooled_dir = results_dir / 'pooled'
pooled_dir.mkdir(parents=True, exist_ok=True)
if len(all_results) > 1:
    maf.pooled_figures(all_results, pooled_dir)
    display(SVG(str(pooled_dir / 'figure1_pooled.svg')))
    display(SVG(str(pooled_dir / 'figure2_pooled.svg')))
    print(f'pooled over {len(all_results)} sessions')

# cross-block control, pooled over every session that has it (needed by report Figure 2D)
control_files = sorted(results_dir.glob('*/controls_results.pkl'))
if control_files:
    controls = [pickle.load(open(p, 'rb')) for p in control_files]
    mcf.figure_controls(controls, pooled_dir)
    print(f'controls pooled over {len(controls)} sessions')

# report figures (Figure 1, Figure 2, Supplementary Figure 1), rebuilt from all sessions so far
if control_files:
    mrf.RESULTS_DIR = results_dir
    mrf.SESSIONS = [r['session_id'] for r in all_results]
    report_dir = results_dir / 'report' / 'figures'
    report_dir.mkdir(parents=True, exist_ok=True)
    mrf.figure_r1(all_results, report_dir)
    mrf.figure_rs1(all_results, report_dir)
    mrf.figure_r2(all_results, report_dir)
    for name in ['figure_R1_subspace_and_decoding', 'figure_R2_alignment', 'figure_S1_dimensionality']:
        display(SVG(str(report_dir / f'{name}.svg')))
    print(f'report figures -> {report_dir} ({len(all_results)} sessions)')
